## Preliminaries

In [ ]:
# Global settings
options(scipen=999)

Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Paths
ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
CODE_PATH <- file.path(ROOT_PATH, "code")
DATA_PATH <- file.path(ROOT_PATH, "data")
OUTPUT_DATA_PATH <- file.path(DATA_PATH, "seasonality_rainfall_percentage")

In [ ]:
# Load utils
source(file.path(CODE_PATH, "snt_utils.r"))

In [ ]:
# List required packages
required_packages <- c(
  "jsonlite",
  "data.table",
  "arrow",
  "glue",
  "sf",
  "httr",
  "reticulate"
)

install_and_load(required_packages)

In [ ]:
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

In [ ]:
# Load SNT config
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
    error = function(e) {
        msg <- paste0("Error while loading configuration", conditionMessage(e))
        cat(msg)
        stop(msg)
    })

msg <- paste0("SNT configuration loaded from  : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME))
log_msg(msg)

COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
era5_dataset <- config_json$SNT_DATASET_IDENTIFIERS$ERA5_DATASET_CLIMATE
dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

print(paste("Country code: ", COUNTRY_CODE))

## Parameters

In [ ]:
# Parameters
top_months_n <- as.integer(4)

if (top_months_n <= 0 || top_months_n > 12) {
  stop("top_months_n must be between 1 and 12")
}

## Load data

In [ ]:
# Load spatial file from dataset
spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")
spatial_data <- get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename)
log_msg(glue("File {spatial_data_filename} successfully loaded from dataset version: {dhis2_dataset}"))

In [ ]:
# Load rainfall data from dataset
rainfall_data_filename <- paste(COUNTRY_CODE, "total_precipitation_monthly.parquet", sep = "_")
original_dt <- get_latest_dataset_file_in_memory(era5_dataset, rainfall_data_filename)
log_msg(glue("File {rainfall_data_filename} successfully loaded from dataset version: {era5_dataset}"))

## Compute rainfall proportions

In [ ]:
# Global variables
admin_level <- "ADM2"
admin_id_col <- paste(admin_level, toupper("id"), sep = "_")
year_col <- "YEAR"
month_col <- "MONTH"
original_values_col <- "MEAN"

admin_data <- st_drop_geometry(spatial_data)
setDT(admin_data)
setDT(original_dt)

integer_cols <- c(year_col, month_col)
original_dt[, (integer_cols) := lapply(.SD, as.integer), .SDcols = integer_cols]

# Aggregate rainfall per month
rainfall_dt <- original_dt[
  ,
  .(
    RAIN_TOTAL = if (all(is.na(get(original_values_col)))) {
      NA_real_
    } else {
      sum(get(original_values_col), na.rm = TRUE)
    }
  ),
  by = c(admin_id_col, year_col, month_col)
]

# Build a full grid of months per admin-year
unique_admin_years <- unique(rainfall_dt[, .SD, .SDcols = c(admin_id_col, year_col)])
all_rows <- unique_admin_years[, .(MONTH = 1:12), by = c(admin_id_col, year_col)]
full_dt <- merge(all_rows, rainfall_dt, by = c(admin_id_col, year_col, month_col), all.x = TRUE)

# Compute yearly totals and proportions
full_dt[, YEAR_TOTAL := sum(RAIN_TOTAL, na.rm = TRUE), by = c(admin_id_col, year_col)]
full_dt[, MONTHS_WITH_DATA := sum(!is.na(RAIN_TOTAL)), by = c(admin_id_col, year_col)]
full_dt[, VALID_YEAR := MONTHS_WITH_DATA == 12 & YEAR_TOTAL > 0]
full_dt[, MONTHLY_PROPORTION := ifelse(VALID_YEAR, RAIN_TOTAL / YEAR_TOTAL, NA_real_)]

# Select top months per admin-year
top_months_dt <- full_dt[
  VALID_YEAR == TRUE,
  head(.SD[order(-RAIN_TOTAL, get(month_col))], top_months_n),
  by = c(admin_id_col, year_col)
]

## Summaries

In [ ]:
# Yearly summaries per admin
yearly_summary_dt <- top_months_dt[
  ,
  .(
    TOP4_RAINFALL_PROPORTION = sum(MONTHLY_PROPORTION, na.rm = TRUE),
    TOP4_RAINFALL_MONTHS = paste(sprintf("%02d", get(month_col)), collapse = ",")
  ),
  by = c(admin_id_col, year_col)
]

yearly_meta_dt <- unique(
  full_dt[
    ,
    .(
      ADM_ID = get(admin_id_col),
      YEAR_VALUE = get(year_col),
      YEAR_TOTAL = YEAR_TOTAL,
      MONTHS_WITH_DATA = MONTHS_WITH_DATA,
      VALID_YEAR = VALID_YEAR
    )
  ]
)
setnames(yearly_meta_dt, c("ADM_ID", "YEAR_VALUE"), c(admin_id_col, year_col))

yearly_summary_dt <- merge(
  yearly_meta_dt,
  yearly_summary_dt,
  by = c(admin_id_col, year_col),
  all.x = TRUE
)

# Compute mode of top-months strings per admin
mode_dt <- yearly_summary_dt[
  !is.na(TOP4_RAINFALL_MONTHS),
  .N,
  by = c(admin_id_col, "TOP4_RAINFALL_MONTHS")
]
setorderv(mode_dt, cols = c(admin_id_col, "N", "TOP4_RAINFALL_MONTHS"), order = c(1, -1, 1))
mode_dt <- mode_dt[
  ,
  .(
    TOP4_RAINFALL_MONTHS_MODE = TOP4_RAINFALL_MONTHS,
    TOP4_RAINFALL_MONTHS_MODE_COUNT = N
  ),
  by = c(admin_id_col)
]

# Summary across years per admin
admin_summary_dt <- yearly_summary_dt[
  VALID_YEAR == TRUE,
  .(
    TOP4_RAINFALL_PROPORTION_MEAN = mean(TOP4_RAINFALL_PROPORTION, na.rm = TRUE),
    TOP4_RAINFALL_PROPORTION_MEDIAN = median(TOP4_RAINFALL_PROPORTION, na.rm = TRUE),
    TOP4_RAINFALL_YEARS_AVAILABLE = sum(!is.na(TOP4_RAINFALL_PROPORTION))
  ),
  by = c(admin_id_col)
]

admin_summary_dt <- merge(admin_summary_dt, mode_dt, by = admin_id_col, all.x = TRUE)
admin_summary_dt[
  ,
  TOP4_RAINFALL_MONTHS_MODE_SHARE := ifelse(
    TOP4_RAINFALL_YEARS_AVAILABLE > 0,
    TOP4_RAINFALL_MONTHS_MODE_COUNT / TOP4_RAINFALL_YEARS_AVAILABLE,
    NA_real_
  )
]

# Attach admin attributes
final_summary_dt <- merge(admin_data, admin_summary_dt, by = admin_id_col, all.x = TRUE)

## Save output

In [ ]:
# Create filenames (same format as rainfall seasonality)
file_stem <- paste(COUNTRY_CODE, "rainfall_proportion", sep = "_")
filename_csv <- glue("{file_stem}.csv")
filename_parquet <- glue("{file_stem}.parquet")

fwrite(final_summary_dt, file.path(OUTPUT_DATA_PATH, filename_csv))
write_parquet(final_summary_dt, file.path(OUTPUT_DATA_PATH, filename_parquet))

log_msg(paste0("Rainfall proportion results saved in folder ", OUTPUT_DATA_PATH))